# Mapa dos fluxos de tráfego aéreo internacional

## Objetivo

Localizar os principais aeroportos, destinos e fluxos origem–destino da base de tráfego internacional dos Estados Unidos. A leitura espacial complementa a análise temporal anterior e ajuda a identificar hubs e corredores de maior volume.

## Contexto e premissas

O mapa usa passageiros e partidas agregados entre 1990 e 2020. Para reduzir poluição visual, são exibidos os 25 maiores aeroportos e os 40 maiores fluxos de cada métrica. Os códigos IATA são associados a coordenadas do pacote `airportsdata`. O mapa é interativo, mas os tiles CartoDB são carregados pela internet.

## A base tem coordenadas suficientes para o mapa?
A base não traz latitude e longitude; ela traz códigos IATA. Vamos verificar a cobertura desses códigos em uma fonte estruturada de aeroportos antes de desenhar os pontos.

In [1]:
from pathlib import Path
import math
import pandas as pd
import folium
import airportsdata

DATASET_HANDLE = "parulpandey/us-international-air-traffic-data"
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".data").exists()), Path.cwd())
OUTPUT_MAP = PROJECT_ROOT / "dashboards/airlines-traffic-map/mapa_fluxos_airlines.html"
TOP_AIRPORTS = 25
TOP_FLOWS = 40


In [2]:
import kagglehub

dataset_dir = Path(kagglehub.dataset_download(DATASET_HANDLE))
passengers = pd.read_csv(dataset_dir / "International_Report_Passengers.csv")
departures = pd.read_csv(dataset_dir / "International_Report_Departures.csv")

print("Passageiros:", passengers.shape)
print("Partidas:", departures.shape)

Passageiros: (680985, 16)
Partidas: (930808, 16)


In [3]:
iata = airportsdata.load("IATA")

codigos = set(passengers["usg_apt"].dropna()) | set(passengers["fg_apt"].dropna())
sem_coordenada = sorted(codigos - set(iata))
print("Códigos IATA distintos:", len(codigos))
print("Códigos sem coordenada:", len(sem_coordenada))
print("Exemplos sem coordenada:", sem_coordenada[:10])

Códigos IATA distintos: 2140
Códigos sem coordenada: 256
Exemplos sem coordenada: ['1B1', '3TX', 'ACU', 'AKR', 'AL5', 'ALY', 'AMF', 'B2C', 'B4C', 'BAK']


**Resultado:** os códigos IATA são usados como chave geográfica. Qualquer código sem correspondência é removido do mapa e contabilizado, evitando inserir coordenadas aproximadas.

## Onde estão os principais aeroportos e destinos?
Agregamos os registros por aeroporto e usamos círculos proporcionais ao volume de passageiros ou partidas.

In [4]:
def agregar_aeroportos(tabela, coluna):
    return (tabela.groupby(coluna, as_index=False)
            .agg(total=("Total", "sum"), registros=("Total", "size"))
            .rename(columns={coluna: "iata"})
            .sort_values("total", ascending=False))

def adicionar_coordenadas(tabela):
    dados = tabela.copy()
    dados["lat"] = dados["iata"].map(lambda x: iata.get(x, {}).get("lat"))
    dados["lon"] = dados["iata"].map(lambda x: iata.get(x, {}).get("lon"))
    return dados.dropna(subset=["lat", "lon"])

top_passenger_origins = adicionar_coordenadas(agregar_aeroportos(passengers, "usg_apt")).head(TOP_AIRPORTS)
top_passenger_destinations = adicionar_coordenadas(agregar_aeroportos(passengers, "fg_apt")).head(TOP_AIRPORTS)
top_departure_origins = adicionar_coordenadas(agregar_aeroportos(departures, "usg_apt")).head(TOP_AIRPORTS)
top_departure_destinations = adicionar_coordenadas(agregar_aeroportos(departures, "fg_apt")).head(TOP_AIRPORTS)

print("Top origem de passageiros:")
display(top_passenger_origins.head(10))
print("Top destino de passageiros:")
display(top_passenger_destinations.head(10))

Top origem de passageiros:


,iata,total,registros,lat,lon
369,JFK,635271483,64122,40.639928,-73.778692
396,LAX,492462404,47760,33.942496,-118.408049
470,MIA,489260654,61939,25.795361,-80.290116
573,ORD,289763281,34952,41.976940,-87.908150
221,EWR,258195184,33024,40.692481,-74.168688
705,SFO,244678358,20618,37.618806,-122.375417
40,ATL,215457577,24676,33.636700,-84.427864
338,IAH,193806073,27225,29.984435,-95.341442
168,DFW,152835097,18856,32.897233,-97.037695
312,HNL,147006062,14694,21.317825,-157.920250


Top destino de passageiros:


,iata,total,registros,lat,lon
574,LHR,328296029,15411,51.4706,-0.46194
1276,YYZ,258191532,43254,43.6772,-79.63060
731,NRT,254928917,16608,35.7647,140.38600
363,FRA,174820916,13763,50.0264,8.54313
186,CDG,154641371,11459,49.0128,2.55000
259,CUN,150177939,30401,21.0365,-86.87710
636,MEX,149869428,17821,19.4363,-99.07210
40,AMS,126092797,9993,52.3086,4.76389
1245,YVR,121361766,17875,49.1939,-123.18400
572,LGW,102037880,9733,51.1481,-0.19028


## Quais são os maiores corredores de tráfego?
As linhas conectam os centroides dos aeroportos de origem e destino. Elas representam uma ligação agregada, não a trajetória geográfica real de cada voo.

In [5]:
def agregar_fluxos(tabela):
    fluxo = (tabela.groupby(["usg_apt", "fg_apt"], as_index=False)
             .agg(total=("Total", "sum"), registros=("Total", "size")))
    fluxo["origem_lat"] = fluxo["usg_apt"].map(lambda x: iata.get(x, {}).get("lat"))
    fluxo["origem_lon"] = fluxo["usg_apt"].map(lambda x: iata.get(x, {}).get("lon"))
    fluxo["destino_lat"] = fluxo["fg_apt"].map(lambda x: iata.get(x, {}).get("lat"))
    fluxo["destino_lon"] = fluxo["fg_apt"].map(lambda x: iata.get(x, {}).get("lon"))
    return fluxo.dropna(subset=["origem_lat", "origem_lon", "destino_lat", "destino_lon"]).sort_values("total", ascending=False)

top_passenger_flows = agregar_fluxos(passengers).head(TOP_FLOWS)
top_departure_flows = agregar_fluxos(departures).head(TOP_FLOWS)
top_passenger_flows.head(10)

,usg_apt,fg_apt,total,registros,origem_lat,origem_lon,destino_lat,destino_lon
8333,JFK,LHR,77012005,2041,40.639928,-73.778692,51.4706,-0.46194
6640,HNL,NRT,54823916,1922,21.317825,-157.920250,35.7647,140.38600
9117,LAX,NRT,40847661,2841,33.942496,-118.408049,35.7647,140.38600
9063,LAX,LHR,39955778,1758,33.942496,-118.408049,51.4706,-0.46194
8211,JFK,CDG,32959031,1428,40.639928,-73.778692,49.0128,2.55000
12308,ORD,LHR,32943258,1272,41.976940,-87.908150,51.4706,-0.46194
9208,LAX,TPE,28340141,1303,33.942496,-118.408049,25.0777,121.23300
9491,LGA,YYZ,27575481,1325,40.777242,-73.872606,43.6772,-79.63060
12449,ORD,YYZ,27485248,2103,41.976940,-87.908150,43.6772,-79.63060
15245,SFO,LHR,26256834,1044,37.618806,-122.375417,51.4706,-0.46194


## Como o mapa permite comparar passageiros, partidas e fluxos?
Criamos camadas independentes para ligar e desligar origens, destinos e rotas. O tamanho dos marcadores é proporcional ao volume, com uma transformação suave para evitar que os maiores hubs dominem todo o mapa.

In [6]:
def raio_proporcional(valor, maximo):
    return 4 + 10 * math.sqrt(valor / maximo)

def adicionar_pontos(mapa, dados, nome, cor, unidade):
    grupo = folium.FeatureGroup(name=nome, show=False)
    maximo = dados["total"].max()
    for _, linha in dados.iterrows():
        aeroporto = iata.get(linha["iata"], {})
        popup = (f"<b>{linha['iata']}</b><br>"
                 f"{aeroporto.get('name', 'Aeroporto não identificado')}<br>"
                 f"{unidade}: {linha['total']:,.0f}<br>"
                 f"Registros: {linha['registros']:,}")
        folium.CircleMarker(location=[linha["lat"], linha["lon"]], radius=raio_proporcional(linha["total"], maximo), color=cor, fill=True, fill_opacity=0.72, popup=popup, tooltip=f"{linha['iata']} — {linha['total']:,.0f}").add_to(grupo)
    grupo.add_to(mapa)

def adicionar_rotas(mapa, dados, nome, cor, unidade):
    grupo = folium.FeatureGroup(name=nome, show=False)
    maximo = dados["total"].max()
    for _, linha in dados.iterrows():
        popup = (f"<b>{linha['usg_apt']} → {linha['fg_apt']}</b><br>"
                 f"{unidade}: {linha['total']:,.0f}<br>"
                 f"Registros: {linha['registros']:,}")
        folium.PolyLine(locations=[[linha["origem_lat"], linha["origem_lon"]], [linha["destino_lat"], linha["destino_lon"]]], color=cor, weight=1.5 + 5 * math.sqrt(linha["total"] / maximo), opacity=0.42, popup=popup, tooltip=f"{linha['usg_apt']} → {linha['fg_apt']} — {linha['total']:,.0f}").add_to(grupo)
    grupo.add_to(mapa)

In [7]:
mapa = folium.Map(location=[38, -55], zoom_start=3, tiles="OpenStreetMap", control_scale=True, prefer_canvas=True)
folium.TileLayer("Cartodb dark_matter", name="Fundo escuro", control=True, show=False).add_to(mapa)

adicionar_pontos(mapa, top_passenger_origins, "Passageiros — origens", "#118DFF", "Passageiros")
adicionar_pontos(mapa, top_passenger_destinations, "Passageiros — destinos", "#E66C37", "Passageiros")
adicionar_rotas(mapa, top_passenger_flows, "Passageiros — maiores fluxos", "#6B007B", "Passageiros")
adicionar_pontos(mapa, top_departure_origins, "Partidas — origens", "#1AAB40", "Partidas")
adicionar_pontos(mapa, top_departure_destinations, "Partidas — destinos", "#D9B300", "Partidas")
adicionar_rotas(mapa, top_departure_flows, "Partidas — maiores fluxos", "#D64550", "Partidas")

folium.LayerControl(collapsed=False).add_to(mapa)
from branca.element import MacroElement, Template

legenda = """<div style=\"position: fixed; bottom: 24px; left: 24px; z-index: 9999; background: white; padding: 10px 12px; border: 1px solid #C8C6C4; border-radius: 6px; font: 12px Segoe UI, sans-serif; color: #252423;\"><b>Camadas</b><br><span style=\"color:#118DFF\">●</span> Passageiros — origens<br><span style=\"color:#E66C37\">●</span> Passageiros — destinos<br><span style=\"color:#1AAB40\">●</span> Partidas — origens<br><span style=\"color:#D9B300\">●</span> Partidas — destinos<br><span style=\"color:#6B007B\">━</span> Rotas de passageiros<br><span style=\"color:#D64550\">━</span> Rotas de partidas</div>"""
mapa.get_root().html.add_child(folium.Element(legenda))
mapa.save(OUTPUT_MAP)
print(f"Mapa salvo em: {OUTPUT_MAP}")
print(f"Camadas de pontos: {TOP_AIRPORTS} origens e {TOP_AIRPORTS} destinos por métrica")
print(f"Camadas de rotas: {TOP_FLOWS} fluxos por métrica")

Mapa salvo em: /Users/danillosantanadearaujo/Documents/Python Scripts/BI Projects/Gerencial/dashboards/airlines-traffic-map/mapa_fluxos_airlines.html
Camadas de pontos: 25 origens e 25 destinos por métrica
Camadas de rotas: 40 fluxos por métrica


/Users/danillosantanadearaujo/Documents/Python Scripts/BI Projects/Gerencial/.venv/lib/python3.13/site-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


## Como interpretar o mapa

- Marcadores maiores indicam maior volume agregado no período.
- Linhas mais espessas representam os maiores fluxos origem–destino.
- Use o controle de camadas para comparar passageiros e partidas.
- As linhas são conexões entre aeroportos, não rotas de voo desenhadas sobre a malha aérea.
- A análise espacial é descritiva; não mede causalidade nem eficiência operacional.

## Próximos aprofundamentos

O mapa atual resume 1990–2020. O próximo passo pode adicionar um seletor temporal por décadas ou anos, mapas separados para 1990, 2000, 2010 e 2019, e uma camada de variação percentual por aeroporto/rota. Para feriados específicos, ainda será necessária uma base diária.